In [1]:
import pandas as pd

fichier = r"C:\Users\LENOVO\Desktop\Nouveau dossier\fe6cb615d161452c_MOHANAD_A4706\fe6cb615d161452c_MOHANAD_A4706\data\NF-UNSW-NB15-v2.csv"

df = pd.read_csv(fichier)
print(len(df), "lignes,", df.shape[1], "colonnes")
print("\nNoms des colonnes :")
print(list(df.columns))

if "Attack" in df.columns:
    print("\nEffectifs par type de trafic :")
    print(df["Attack"].value_counts().to_string())

2390275 lignes, 45 colonnes

Noms des colonnes :
['IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'Label', 'Attack']

Effectifs par type de trafic :
Attack
Benign            2295222
Exploits  

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

fichier = r"C:\Users\LENOVO\Desktop\Nouveau dossier\fe6cb615d161452c_MOHANAD_A4706\fe6cb615d161452c_MOHANAD_A4706\data\NF-UNSW-NB15-v2.csv"   # <-- votre vrai chemin
assert Path(fichier).exists(), "Fichier introuvable : vérifiez le chemin."

df = pd.read_csv(fichier)

# Colonnes d'identification, absentes de CIC-IDS2017
ident = ["IPV4_SRC_ADDR", "L4_SRC_PORT", "IPV4_DST_ADDR"]
X = df.drop(columns=ident)
num = X.select_dtypes(include=[np.number])

print("Lignes :", len(X))
print("Doublons :", int(X.duplicated().sum()))
print("Lignes avec infini :", int(np.isinf(num).any(axis=1).sum()))
print("Lignes avec vide :", int(num.isna().any(axis=1).sum()))

constantes = [c for c in num.columns if num[c].nunique() <= 1]
print("Colonnes constantes :", constantes)
print("Colonnes non numériques :", list(X.select_dtypes(exclude=[np.number]).columns))
print("Valeurs de Label :", sorted(df["Label"].unique()))

Lignes : 2390275
Doublons : 1525678
Lignes avec infini : 0
Lignes avec vide : 0
Colonnes constantes : []
Colonnes non numériques : ['Attack']
Valeurs de Label : [np.int64(0), np.int64(1)]


In [4]:
feats = df.drop(columns=ident + ["Label", "Attack"])
h = pd.util.hash_pandas_object(feats, index=False)   # une « empreinte » par ligne
t = pd.DataFrame({"h": h, "Label": df["Label"], "Attack": df["Attack"]})

# 1. Doublons par type de trafic (mêmes mesures ET même type)
dup = t.duplicated(subset=["h", "Attack"])
res = pd.DataFrame({
    "Lignes": t["Attack"].value_counts(),
    "Doublons": t[dup]["Attack"].value_counts(),
}).fillna(0).astype(int)
res["Restantes"] = res["Lignes"] - res["Doublons"]
res["% doublons"] = (100 * res["Doublons"] / res["Lignes"]).round(1)
print(res.sort_values("Lignes", ascending=False).to_string())

# 2. Profils identiques avec des étiquettes contradictoires (normal ET attaque)
n = t.groupby("h")["Label"].nunique()
print("\nProfils avec étiquettes contradictoires :", int((n > 1).sum()))
print("Lignes concernées :", int(t["h"].isin(n[n > 1].index).sum()))

                 Lignes  Doublons  Restantes  % doublons
Attack                                                  
Benign          2295222   1473993     821229        64.2
Exploits          31551      7818      23733        24.8
Fuzzers           22310     14905       7405        66.8
Generic           16560     12296       4264        74.3
Reconnaissance    12779     11636       1143        91.1
DoS                5794      1946       3848        33.6
Analysis           2299      1537        762        66.9
Backdoor           2169      1452        717        66.9
Shellcode          1427         0       1427         0.0
Worms               164        95         69        57.9

Profils avec étiquettes contradictoires : 0
Lignes concernées : 0


In [1]:
import pandas as pd

fichier = r"C:\Users\LENOVO\Documents\MachineLearningCSV\MachineLearningCVE\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"

y = pd.read_csv(fichier, encoding="latin1",
                usecols=lambda c: c.strip() == "Label")
y.columns = ["Label"]
etiq = y["Label"]

changements = int((etiq != etiq.shift()).sum() - 1)
print("Lignes :", len(etiq))
print("Changements d'étiquette d'une ligne à la suivante :", changements)

Lignes : 286467
Changements d'étiquette d'une ligne à la suivante : 5006


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

dossier = r"C:\Users\LENOVO\Documents\MachineLearningCSV\MachineLearningCVE"                       # <-- vos 8 CSV
sortie  = r"C:\Users\LENOVO\Documents\MachineLearningCSV\dataPreprocessing"        # <-- où sauvegarder

fichiers = sorted(Path(dossier).glob("*.pcap_ISCX*"))
print(len(fichiers), "fichiers trouvés")
assert len(fichiers) == 8, "Il devrait y avoir 8 fichiers."

constantes = ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk',
              'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk',
              'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

# 1. Lecture : on garde l'ordre des lignes de chaque fichier
parts = []
for f in fichiers:
    df = pd.read_csv(f, encoding="latin1", low_memory=False)
    df.columns = df.columns.str.strip()
    df = df.drop(columns=constantes)
    df["Fichier"] = f.name.replace(".pcap_ISCX.csv", "")
    df["Ordre"] = np.arange(len(df))
    parts.append(df)
data = pd.concat(parts, ignore_index=True)
del parts

# 2. Nettoyage du nom abîmé et retrait des doublons sur toute la semaine
data["Label"] = data["Label"].str.replace("ï¿½", "-", regex=False)
mesures = [c for c in data.columns if c not in ("Fichier", "Ordre", "Label")]

contradictions = int(data.duplicated(subset=mesures).sum()
                     - data.duplicated(subset=mesures + ["Label"]).sum())
avant = len(data)
data = data.drop_duplicates(subset=mesures + ["Label"], keep="first")
print(f"Doublons retirés (semaine entière) : {avant - len(data):,} sur {avant:,}")
print("Profils identiques avec étiquettes contradictoires :", contradictions)

# 3. Séparation dans le temps, fichier par fichier
rang = data.groupby("Fichier")["Ordre"].rank(pct=True)
data["Partie"] = np.where(rang <= 0.70, "train",
                  np.where(rang <= 0.71, "ecart", "test"))

tableau = pd.crosstab(data["Label"], data["Partie"], margins=True)
print("\n", tableau[["train", "ecart", "test", "All"]].to_string())

# 4. Sauvegarde (l'écart est jeté)
Path(sortie).mkdir(parents=True, exist_ok=True)
data[data["Partie"] == "train"].drop(columns="Partie").to_pickle(Path(sortie) / "cicids_train.pkl")
data[data["Partie"] == "test"].drop(columns="Partie").to_pickle(Path(sortie) / "cicids_test.pkl")
print("\nFichiers sauvegardés dans", sortie)

8 fichiers trouvés
Doublons retirés (semaine entière) : 308,381 sur 2,830,743
Profils identiques avec étiquettes contradictoires : 698

 Partie                        train  ecart    test      All
Label                                                      
BENIGN                      1405976  21174  669334  2096484
Bot                            1509      8     436     1953
DDoS                         101020   1908   25088   128016
DoS GoldenEye                  1122      0    9164    10286
DoS Hulk                     172849      0       0   172849
DoS Slowhttptest               5228      0       0     5228
DoS slowloris                  5385      0       0     5385
FTP-Patator                    5933      0       0     5933
Heartbleed                        0      0      11       11
Infiltration                     32      1       3       36
PortScan                      62511   2085   26223    90819
SSH-Patator                    1942     48    1229     3219
Web Attack - Brute Forc

In [3]:
rang = data.groupby(["Fichier", "Label"])["Ordre"].rank(pct=True)
data["Partie"] = np.where(rang <= 0.70, "train",
                  np.where(rang <= 0.71, "ecart", "test"))

tableau = pd.crosstab(data["Label"], data["Partie"], margins=True)
print(tableau[["train", "ecart", "test", "All"]].to_string())

data[data["Partie"] == "train"].drop(columns="Partie").to_pickle(Path(sortie) / "cicids_train.pkl")
data[data["Partie"] == "test"].drop(columns="Partie").to_pickle(Path(sortie) / "cicids_test.pkl")
print("Fichiers sauvegardés (anciens remplacés)")

Partie                        train  ecart    test      All
Label                                                      
BENIGN                      1467536  20963  607985  2096484
Bot                            1367     19     567     1953
DDoS                          89611   1280   37125   128016
DoS GoldenEye                  7200    103    2983    10286
DoS Hulk                     120994   1728   50127   172849
DoS Slowhttptest               3659     52    1517     5228
DoS slowloris                  3769     54    1562     5385
FTP-Patator                    4153     59    1721     5933
Heartbleed                        7      0       4       11
Infiltration                     25      0      11       36
PortScan                      63573    908   26338    90819
SSH-Patator                    2253     32     934     3219
Web Attack - Brute Force       1029     14     427     1470
Web Attack - Sql Injection       14      0       7       21
Web Attack - XSS                456     

In [4]:
# 1. Profils ambigus : mêmes mesures, étiquettes différentes
mesures = [c for c in data.columns if c not in ("Fichier", "Ordre", "Label", "Partie")]
h = pd.util.hash_pandas_object(data[mesures], index=False)
ambigu = data.groupby(h)["Label"].transform("nunique") > 1
print("Lignes ambiguës retirées :", int(ambigu.sum()))

# 2. On enlève les ambigus et la zone tampon
data = data[~ambigu & (data["Partie"] != "ecart")].copy()

# 3. Infinis -> vides, puis remplissage par la médiane de la partie "train" uniquement
data[mesures] = data[mesures].replace([np.inf, -np.inf], np.nan)
mediane = data.loc[data["Partie"] == "train", mesures].median()
data[mesures] = data[mesures].fillna(mediane)
print("Valeurs vides restantes :", int(data[mesures].isna().sum().sum()))

# 4. Sauvegarde
train = data[data["Partie"] == "train"].drop(columns="Partie")
test  = data[data["Partie"] == "test"].drop(columns="Partie")
train.to_pickle(Path(sortie) / "cicids_train_propre.pkl")
test.to_pickle(Path(sortie) / "cicids_test_propre.pkl")
mediane.to_pickle(Path(sortie) / "mediane_train.pkl")
print("Train :", len(train), "| Test :", len(test))

Lignes ambiguës retirées : 1396
Valeurs vides restantes : 0
Train : 1764404 | Test : 731345


In [5]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

a_retirer = []          # plus tard, on testera ["Destination Port"]
cols = [c for c in mesures if c not in a_retirer]

# 1. Entraînement : trafic normal limité à 300 000 lignes, toutes les attaques gardées
benin = train[train["Label"] == "BENIGN"].sample(n=300_000, random_state=42)
train_ech = pd.concat([benin, train[train["Label"] != "BENIGN"]])
Xtr, ytr = train_ech[cols].astype("float32"), train_ech["Label"]
Xte, yte = test[cols].astype("float32"), test["Label"]
print("Lignes d'entraînement :", len(Xtr))

t0 = time.perf_counter()
modele = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
modele.fit(Xtr, ytr)
temps_train = time.perf_counter() - t0

# 2. Prédiction sur tout le test
t0 = time.perf_counter()
pred = modele.predict(Xte)
temps_pred = time.perf_counter() - t0

# 3. Résultats par classe
rapport = pd.DataFrame(classification_report(yte, pred, digits=3,
                       zero_division=0, output_dict=True)).T
print(rapport.round(3).to_string())

# 4. Question simple : attaque ou normal ?
vrai = (yte != "BENIGN").values
prevu = pred != "BENIGN"
print("\nAttaques détectées (rappel)       :", round((vrai & prevu).sum() / vrai.sum(), 4))
print("Trafic normal accusé à tort (FPR) :", round((~vrai & prevu).sum() / (~vrai).sum(), 5))
print("Temps d'entraînement (s)          :", round(temps_train, 1))
print("Temps de prédiction par flux (µs) :", round(1e6 * temps_pred / len(Xte), 1))

# 5. Sauvegarde du tableau pour l'article
resultats = Path(sortie).parent / "results"
resultats.mkdir(exist_ok=True)
rapport.to_csv(resultats / "base_rf_sans_selection.csv")

Lignes d'entraînement : 597528
                            precision  recall  f1-score     support
BENIGN                          0.996   1.000     0.998  607947.000
Bot                             0.616   0.122     0.203     567.000
DDoS                            1.000   0.998     0.999   37124.000
DoS GoldenEye                   0.952   0.971     0.961    2983.000
DoS Hulk                        0.999   0.954     0.976   50108.000
DoS Slowhttptest                0.836   0.856     0.846    1517.000
DoS slowloris                   0.952   0.794     0.866    1562.000
FTP-Patator                     0.999   0.991     0.995    1721.000
Heartbleed                      1.000   0.750     0.857       4.000
Infiltration                    1.000   0.818     0.900      11.000
PortScan                        0.962   0.997     0.979   26243.000
SSH-Patator                     0.998   0.967     0.982     934.000
Web Attack - Brute Force        0.705   0.738     0.721     427.000
Web Attack - Sql 

In [6]:
vraies = ["Bot", "Web Attack - XSS", "Web Attack - Brute Force",
          "DoS slowloris", "DoS Slowhttptest"]
masque = yte.isin(vraies).values

vrai = pd.Series(yte[masque].to_numpy(), name="Vrai").str.replace("Web Attack - ", "Web ", regex=False)
prevu = pd.Series(pred[masque], name="Prédit").str.replace("Web Attack - ", "Web ", regex=False)

conf = pd.crosstab(vrai, prevu)
conf = conf.loc[:, conf.sum() > 0]          # on garde seulement les colonnes utilisées

print("Nombre de flux :")
print(conf.to_string())
print("\nEn pourcentage de chaque ligne :")
print((100 * conf.div(conf.sum(axis=1), axis=0)).round(1).to_string())

Nombre de flux :
Prédit            BENIGN  Bot  DoS Hulk  DoS Slowhttptest  DoS slowloris  PortScan  Web Brute Force  Web Sql Injection  Web XSS
Vrai                                                                                                                           
Bot                  498   69         0                 0              0         0                0                  0        0
DoS Slowhttptest     162    0         0              1298             55         0                1                  0        1
DoS slowloris        105    0         1               214           1241         0                0                  1        0
Web Brute Force        2    0         0                 0              0         3              315                  0      107
Web XSS               14    0         0                 0              0         1              125                  0       50

En pourcentage de chaque ligne :
Prédit            BENIGN   Bot  DoS Hulk  DoS Slowhtt

In [7]:
cols2 = [c for c in mesures if c != "Destination Port"]
Xtr2 = train_ech[cols2].astype("float32")
Xte2 = test[cols2].astype("float32")

modele2 = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
modele2.fit(Xtr2, ytr)
pred2 = modele2.predict(Xte2)

rapport2 = pd.DataFrame(classification_report(yte, pred2, digits=3,
                        zero_division=0, output_dict=True)).T

classes = [c for c in rapport.index if c in set(yte.unique())]
comp = pd.DataFrame({
    "F1 avec port":     rapport.loc[classes, "f1-score"],
    "F1 sans port":     rapport2.loc[classes, "f1-score"],
    "Rappel avec port": rapport.loc[classes, "recall"],
    "Rappel sans port": rapport2.loc[classes, "recall"],
    "Test":             rapport.loc[classes, "support"].astype(int),
}).round(3)
comp["Écart F1"] = (comp["F1 sans port"] - comp["F1 avec port"]).round(3)
print(comp.to_string())

vrai = (yte != "BENIGN").values
prevu2 = pred2 != "BENIGN"
print("\nAttaques détectées, sans port (rappel) :", round((vrai & prevu2).sum() / vrai.sum(), 4))
print("Trafic normal accusé à tort, sans port :", round((~vrai & prevu2).sum() / (~vrai).sum(), 5))

comp.to_csv(Path(sortie).parent / "results" / "comparaison_avec_sans_port.csv")

                            F1 avec port  F1 sans port  Rappel avec port  Rappel sans port    Test  Écart F1
BENIGN                             0.998         0.997             1.000             0.999  607947    -0.001
Bot                                0.203         0.178             0.122             0.132     567    -0.025
DDoS                               0.999         0.999             0.998             0.998   37124     0.000
DoS GoldenEye                      0.961         0.958             0.971             0.970    2983    -0.003
DoS Hulk                           0.976         0.968             0.954             0.941   50108    -0.008
DoS Slowhttptest                   0.846         0.801             0.856             0.856    1517    -0.045
DoS slowloris                      0.866         0.873             0.794             0.784    1562     0.007
FTP-Patator                        0.995         0.994             0.991             0.989    1721    -0.001
Heartbleed         

In [8]:
classes_ok = [c for c in rapport.index
              if c in set(yte.unique()) and (yte == c).sum() >= 100]
variantes = {"avec port": mesures,
             "sans port": [c for c in mesures if c != "Destination Port"]}
vrai_bin = (yte != "BENIGN").values

lignes = []
for graine in range(5):
    benin = train[train["Label"] == "BENIGN"].sample(n=300_000, random_state=graine)
    tr = pd.concat([benin, train[train["Label"] != "BENIGN"]])
    for nom, cs in variantes.items():
        m = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=graine)
        m.fit(tr[cs].astype("float32"), tr["Label"])
        p = m.predict(test[cs].astype("float32"))
        rep = classification_report(yte, p, output_dict=True, zero_division=0)
        ligne = {"variante": nom, "graine": graine,
                 "rappel_attaque": (vrai_bin & (p != "BENIGN")).sum() / vrai_bin.sum(),
                 "FPR": ((~vrai_bin) & (p != "BENIGN")).sum() / (~vrai_bin).sum()}
        for c in classes_ok:
            ligne["F1 " + c] = rep[c]["f1-score"]
        lignes.append(ligne)
        print("fait :", nom, "graine", graine)
        del m, p

res = pd.DataFrame(lignes)
res.to_csv(Path(sortie).parent / "results" / "repetitions_avec_sans_port.csv", index=False)

moy = res.groupby("variante").mean(numeric_only=True).drop(columns="graine").T
std = res.groupby("variante").std(numeric_only=True).drop(columns="graine").T
print((moy.round(4).astype(str) + " ± " + std.round(4).astype(str)).to_string())

fait : avec port graine 0
fait : sans port graine 0
fait : avec port graine 1
fait : sans port graine 1
fait : avec port graine 2
fait : sans port graine 2
fait : avec port graine 3
fait : sans port graine 3
fait : avec port graine 4
fait : sans port graine 4
variante                           avec port        sans port
rappel_attaque               0.9835 ± 0.0005  0.9775 ± 0.0009
FPR                             0.0003 ± 0.0      0.001 ± 0.0
F1 BENIGN                    0.9982 ± 0.0001  0.9972 ± 0.0001
F1 Bot                       0.2038 ± 0.0029  0.1782 ± 0.0024
F1 DDoS                      0.9989 ± 0.0001  0.9988 ± 0.0001
F1 DoS GoldenEye             0.9608 ± 0.0012  0.9598 ± 0.0015
F1 DoS Hulk                  0.9758 ± 0.0006  0.9691 ± 0.0011
F1 DoS Slowhttptest          0.8607 ± 0.0043  0.8096 ± 0.0094
F1 DoS slowloris             0.8817 ± 0.0053  0.8766 ± 0.0041
F1 FTP-Patator               0.9956 ± 0.0007  0.9947 ± 0.0005
F1 PortScan                  0.9766 ± 0.0016  0.9793 ± 0.0

In [10]:
###Affichage des fausses alertes

In [9]:
fpr = res.groupby("variante")["FPR"].agg(["mean", "std"]) * 100
print(fpr.round(4).to_string())

             mean     std
variante                 
avec port  0.0282  0.0024
sans port  0.0981  0.0023
